In [ ]:
import glob
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import torchmetrics.functional

from dataset import OptionsDataModule
from model import OptionNetModule

warnings.filterwarnings("ignore")

In [ ]:
# --- Constants ---
CHECKPOINT_PATH = './checkpoints/run_20260217_234449_749029/epoch=299-step=43200.ckpt'
DATA_DIR = "./data/108105"
SOFR_PATH = "./data/sofr.csv"
BATCH_SIZE = 512
SELECTED_DATE = "2025-08-29"  # Set to None to use latest available date
CP_FLAG = "C"
RISK_FREE_RATE = 0.04  # fallback SOFR when missing
DIVIDEND_YIELD_FALLBACK = 0.0  # fallback dividend yield when missing
RATE_FALLBACK = RISK_FREE_RATE  # fallback short rate when missing


In [ ]:
def choose_device():
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def expected_raw_feature_count(model):
    first_linear = model.model.model[0]
    return int(first_linear.in_features) + 1

In [ ]:
print("Loading Data Module...")
data_module = OptionsDataModule(DATA_DIR, sofr_path=SOFR_PATH, batch_size=BATCH_SIZE)
data_module.setup("fit")

print(f"Loading Model from {CHECKPOINT_PATH}...")
model = OptionNetModule.load_from_checkpoint(CHECKPOINT_PATH)
model.eval()

device = choose_device()
model = model.to(device)

In [ ]:
print("Loading raw data for selected date...")
raw_df = pd.concat(
    [pd.read_csv(path) for path in sorted(glob.glob(f"{DATA_DIR}/*.csv"))],
    ignore_index=True,
)
raw_df["date"] = pd.to_datetime(raw_df["date"])

if "cp_flag" in raw_df.columns:
    raw_df = raw_df[raw_df["cp_flag"] == CP_FLAG].copy()

if "T" not in raw_df.columns or "S" not in raw_df.columns or "K" not in raw_df.columns:
    raise ValueError("Input CSV files must contain columns: T, S, K.")

# Match dataset.py constraints: keep only maturities <= 365 days and in-range moneyness.
raw_df = raw_df[(raw_df["T"] > 14) & (raw_df["T"] <= 365)].copy()
raw_df["M"] = raw_df["S"] / raw_df["K"].clip(lower=1e-8)
raw_df = raw_df[(raw_df["M"] >= 0.5) & (raw_df["M"] <= 1.15)].copy()

sofr_df = pd.read_csv(SOFR_PATH)
sofr_df["date"] = pd.to_datetime(sofr_df["date"])
sofr_df["sofr"] = sofr_df["sofr"] / 100.0

raw_df = pd.merge(raw_df, sofr_df[["date", "sofr"]], on="date", how="left")
raw_df["sofr"] = raw_df["sofr"].fillna(RISK_FREE_RATE)
raw_df["T"] = raw_df["T"] / 365.0
raw_df["vix"] = raw_df["vix"] / 100.0

if raw_df.empty:
    raise ValueError(f"No rows remain under {DATA_DIR} after dataset-style filtering.")

target_date = pd.to_datetime(SELECTED_DATE) if SELECTED_DATE else raw_df["date"].max()
daily = raw_df[raw_df["date"] == target_date].copy()
if daily.empty:
    available = raw_df["date"].dt.date.drop_duplicates().sort_values().astype(str).tail(10).tolist()
    raise ValueError(
        f"No rows found for SELECTED_DATE={target_date.date()}. "
        f"Last available dates: {available}"
    )

S_fixed = float(daily["S"].median())
vix_fixed = float(daily["vix"].median())
sofr_fixed = float(daily["sofr"].median())
dividend_yield_fixed = (
    float(daily["dividend_yield"].median())
    if "dividend_yield" in daily.columns
    else DIVIDEND_YIELD_FALLBACK
)
rate_fixed = float(daily["rate"].median()) if "rate" in daily.columns else RATE_FALLBACK

raw_feature_count = expected_raw_feature_count(model)
print(f"Model expects {raw_feature_count} raw features")

if raw_feature_count not in (4, 7, 9, 10, 11, 12):
    raise ValueError(
        f"Unsupported raw feature count {raw_feature_count}. "
        "Expected one of 4, 7, 9, 10, 11, 12."
    )

include_hv = raw_feature_count in (9, 10, 11, 12)
include_sofr = raw_feature_count in (7, 10, 11, 12)
include_dividend_yield = raw_feature_count in (7, 11, 12)
include_rate = raw_feature_count in (7, 12)

hv_cols = ["hv_10", "hv_14", "hv_30", "hv_60", "hv_91"]
h_vol_values = {}
if include_hv:
    missing_hv = [col for col in hv_cols if col not in daily.columns]
    if missing_hv:
        raise ValueError(f"Missing HV columns required by model: {missing_hv}")
    h_vol_values = daily[hv_cols].median().astype(float).to_dict()
    if any(pd.isna(v) for v in h_vol_values.values()):
        raise ValueError(f"Historical vol is missing for date {target_date.date()} in columns {hv_cols}.")

if pd.isna(vix_fixed):
    raise ValueError(f"VIX is missing for date {target_date.date()}.")
if include_sofr and pd.isna(sofr_fixed):
    sofr_fixed = RISK_FREE_RATE
if include_dividend_yield and pd.isna(dividend_yield_fixed):
    dividend_yield_fixed = DIVIDEND_YIELD_FALLBACK
if include_rate and pd.isna(rate_fixed):
    rate_fixed = RATE_FALLBACK

print(
    f"Using date={target_date.date()}, S={S_fixed:.2f}, VIX={vix_fixed:.4f}, "
    f"SOFR={sofr_fixed:.4f}, DIVIDEND_YIELD={dividend_yield_fixed:.4f}, RATE={rate_fixed:.4f}"
)
if h_vol_values:
    print("Historical vol snapshot:", h_vol_values)

# --- Generate Grid ---
print("Generating Grid...")
moneyness_values = np.linspace(0.5, 1.15, 30)
T_years = np.linspace(15 / 365.0, 365.0 / 365.0, 30)

# Create meshgrid
moneyness_grid, T_grid = np.meshgrid(moneyness_values, T_years)
K_grid = S_fixed / moneyness_grid

# Flatten for model input
K_flat = K_grid.flatten()
T_flat = T_grid.flatten()

feature_blocks = [
    torch.full((len(K_flat), 1), S_fixed, dtype=torch.float32),
    torch.tensor(K_flat, dtype=torch.float32).reshape(-1, 1),
    torch.tensor(T_flat, dtype=torch.float32).reshape(-1, 1),
    torch.full((len(K_flat), 1), vix_fixed, dtype=torch.float32),
]

if include_hv:
    hv_tensors = [
        torch.full((len(K_flat), 1), float(h_vol_values[col]), dtype=torch.float32)
        for col in hv_cols
    ]
    feature_blocks.extend(hv_tensors)

if include_sofr:
    feature_blocks.append(torch.full((len(K_flat), 1), sofr_fixed, dtype=torch.float32))
if include_dividend_yield:
    feature_blocks.append(torch.full((len(K_flat), 1), dividend_yield_fixed, dtype=torch.float32))
if include_rate:
    feature_blocks.append(torch.full((len(K_flat), 1), rate_fixed, dtype=torch.float32))

x_input = torch.cat(feature_blocks, dim=1).to(device)
if x_input.shape[1] != raw_feature_count:
    raise ValueError(
        f"Built {x_input.shape[1]} input features, but model expects {raw_feature_count}."
    )

print("Predicting Prices...")
preds, greeks = model(x_input)

predicted_prices = (preds.detach().cpu().numpy().flatten() * K_flat)

delta_grid = greeks["delta"].detach().cpu().numpy().reshape(K_grid.shape)
gamma_grid = greeks["gamma"].detach().cpu().numpy().reshape(K_grid.shape)
theta_grid = greeks["theta"].detach().cpu().numpy().reshape(K_grid.shape)
vega_grid = (greeks["vega"].detach().cpu().numpy()).reshape(K_grid.shape)

price_grid = np.array(predicted_prices).reshape(K_grid.shape)


def plot_surface(z_grid, title, zaxis_title, colorbar_title):
    fig = go.Figure(data=[
        go.Surface(
            x=moneyness_grid,
            y=T_grid,
            z=z_grid,
            colorscale='Viridis',
            colorbar_title=colorbar_title,
            opacity=0.9,
            hovertemplate=(
                "Moneyness (S/K): %{x:.4f}<br>" +
                "Time (Years): %{y:.2f}<br>" +
                f"{zaxis_title}: %{{z:.6f}}<extra></extra>"
            )
        ),
    ])

    fig.update_layout(
        title={
            'text': title,
            'y': 0.9,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top'
        },
        scene=dict(
            xaxis_title='Moneyness (S/K)',
            yaxis_title='Time to Maturity (Years)',
            zaxis_title=zaxis_title,
            aspectratio=dict(x=1, y=1, z=0.6),
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=0.5)
            )
        ),
        width=1000,
        height=700,
        margin=dict(l=50, r=50, b=0, t=50)
    )
    fig.show()


meta = (
    f"date={target_date.date()}, S={S_fixed:.2f}, VIX={vix_fixed:.4f}, "
    f"SOFR={sofr_fixed:.4f}, DIVIDEND_YIELD={dividend_yield_fixed:.4f}, RATE={rate_fixed:.4f}"
)

print("Plotting Surface...")
plot_surface(
    price_grid,
    f"Price Surface ({meta})",
    'Price',
    'Predicted Price'
)

print("Plotting Delta Surface...")
plot_surface(
    delta_grid,
    f"Delta Surface ({meta})",
    'Delta',
    'Delta'
)

print("Plotting Gamma Surface...")
plot_surface(
    gamma_grid,
    f"Gamma Surface ({meta})",
    'Gamma',
    'Gamma'
)

print("Plotting Theta Surface...")
plot_surface(
    theta_grid,
    f"Theta Surface ({meta})",
    'Theta',
    'Theta'
)

print("Plotting Vega Surface...")
plot_surface(
    vega_grid,
    f"Vega Surface ({meta})",
    'Vega',
    'Vega'
)


In [ ]:
print("Collecting validation/testing price errors...")
torch.set_grad_enabled(True)

import torchmetrics

MONEYNESS_BINS = [0.5, 0.90, 0.95, 1.05, 1.10, 1.15] #[0.0, 0.90, 0.97, 1.03, 1.10, np.inf]
MONEYNESS_LABELS = ["DOTM (0.5-1.9)", "OTM (0.9-0.95)", "ATM (0.95-1.05)", "ITM (1.05-1.10)", "UITM (1.10-inf)"]


def summarize_errors(split_df, split_name):
    errors = split_df["error"].values
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    mape = np.nanmean(split_df["abs_pct_error"].values)
    print(
        f"{split_name}: n={len(errors):,}, mean={errors.mean():.4f}, std={errors.std():.4f}, "
        f"MAE={mae:.4f}, RMSE={rmse:.4f}, MAPE={mape:.2f}%"
    )


def collect_error_frame(loader, split_name):
    moneyness_parts = []
    k_parts = []
    t_days_parts = []
    real_prices = []
    model_prices = []

    real_batch_t = []
    model_batch_t = []

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        preds_norm, _ = model(xb)
        s = xb[:, 0:1]
        k = xb[:, 1:2]

        real_batch_t.append((yb).detach().cpu())
        model_batch_t.append((preds_norm * k).detach().cpu())

        real_batch = (yb).detach().cpu().numpy().ravel()
        model_batch = (preds_norm * k).detach().cpu().numpy().ravel()
        moneyness_batch = (s / k).detach().cpu().numpy().ravel()
        k_batch = k.detach().cpu().numpy().ravel()
        t_days_batch = (xb[:, 2:3] * 365.0).detach().cpu().numpy().ravel()

        real_prices.append(real_batch)
        model_prices.append(model_batch)
        moneyness_parts.append(moneyness_batch)
        k_parts.append(k_batch)
        t_days_parts.append(t_days_batch)

    if not real_prices:
        raise ValueError(f"No rows available for {split_name} set.")

    print(f"R2: {torchmetrics.functional.r2_score(torch.cat(model_batch_t), torch.cat(real_batch_t))}")

    real_prices = np.concatenate(real_prices)
    model_prices = np.concatenate(model_prices)
    moneyness = np.concatenate(moneyness_parts)
    k_values = np.concatenate(k_parts)
    t_days_values = np.concatenate(t_days_parts)

    abs_error = np.abs(model_prices - real_prices)
    abs_pct_error = np.where(np.abs(real_prices) > 1e-8, (abs_error / np.abs(real_prices)) * 100.0, np.nan)

    frame = pd.DataFrame(
        {
            "split": split_name,
            "real_price": real_prices,
            "model_price": model_prices,
            "error": model_prices - real_prices,
            "abs_error": abs_error,
            "abs_pct_error": abs_pct_error,
            "moneyness": moneyness,
            "K": k_values,
            "T_days": t_days_values,
            "difference": real_prices - model_prices,
        }
    )
    frame["moneyness_bucket"] = pd.Categorical(
        pd.cut(frame["moneyness"], bins=MONEYNESS_BINS, labels=MONEYNESS_LABELS, include_lowest=True),
        categories=MONEYNESS_LABELS,
        ordered=True,
    )

    summarize_errors(frame, split_name)
    return frame


val_df = collect_error_frame(data_module.val_dataloader(), "Validation")
test_df = collect_error_frame(data_module.test_dataloader(), "Testing")
all_errors = pd.concat([val_df, test_df], ignore_index=True)
surface_df = all_errors[["split", "K", "T_days", "difference", "moneyness"]].copy()

bucket_metrics = (
    all_errors.groupby(["split", "moneyness_bucket"], observed=True)
    .agg(
        n=("error", "size"),
        mean_error=("error", "mean"),
        std_error=("error", "std"),
        mae=("abs_error", "mean"),
        rmse=("error", lambda x: float(np.sqrt(np.mean(np.square(x))))),
        mape=("abs_pct_error", "mean"),
        mean_moneyness=("moneyness", "mean"),
    )
    .reset_index()
)

print("\nError metrics by moneyness bucket:")
print(bucket_metrics.to_string(index=False))

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=val_df["error"],
    name="Validation Error",
    opacity=0.65,
    histnorm="percent",
    nbinsx=80,
    marker_color="#1f77b4",
))
fig.add_trace(go.Histogram(
    x=test_df["error"],
    name="Testing Error",
    opacity=0.65,
    histnorm="percent",
    nbinsx=80,
    marker_color="#ff7f0e",
))

fig.add_vline(x=0.0, line_dash="dash", line_color="black")
fig.update_layout(
    title="Error Distribution: Model Price - Real Price (Validation vs Testing)",
    xaxis_title="Pricing Error",
    yaxis_title="Percent of Samples (%)",
    barmode="overlay",
    template="plotly_white",
    width=1000,
    height=500,
)
fig.show()

mae_fig = go.Figure()
for split_name, color in [("Validation", "#1f77b4"), ("Testing", "#ff7f0e")]:
    split_metrics = bucket_metrics[bucket_metrics["split"] == split_name]
    mae_fig.add_trace(
        go.Bar(
            x=split_metrics["moneyness_bucket"],
            y=split_metrics["mae"],
            name=f"{split_name} MAE",
            marker_color=color,
            opacity=0.85,
            offsetgroup=f"{split_name}_mae",
            legendgroup=split_name,
        )
    )
    mae_fig.add_trace(
        go.Bar(
            x=split_metrics["moneyness_bucket"],
            y=split_metrics["mape"],
            name=f"{split_name} MAPE",
            marker=dict(color=color, pattern=dict(shape="/")),
            opacity=0.5,
            offsetgroup=f"{split_name}_mape",
            legendgroup=split_name,
            yaxis="y2",
        )
    )
    mae_fig.add_trace(
        go.Scatter(
            x=split_metrics["moneyness_bucket"],
            y=split_metrics["rmse"],
            name=f"{split_name} RMSE",
            mode="lines+markers",
            marker=dict(color=color, size=8, symbol="diamond"),
            line=dict(color=color, dash="dot"),
            legendgroup=split_name,
        )
    )

mae_fig.update_layout(
    title="Pricing Error by Moneyness Bucket (Validation vs Testing)",
    xaxis_title="Moneyness Bucket (S/K)",
    yaxis=dict(title="Price Error (MAE / RMSE)", rangemode="tozero"),
    yaxis2=dict(title="MAPE (%)", overlaying="y", side="right", rangemode="tozero"),
    barmode="group",
    template="plotly_white",
    width=1100,
    height=550,
)
mae_fig.show()

In [ ]:
if surface_df.empty:
    raise ValueError("surface_df is empty. Run the error collection cell first.")

pivot_diff = (
    surface_df.pivot_table(index='T_days', columns='moneyness', values='difference', aggfunc='mean')
    .sort_index()
    .sort_index(axis=1)
)

if pivot_diff.empty:
    raise ValueError("No data available to render the heatmap.")

heatmap = go.Figure(
    data=go.Heatmap(
        z=pivot_diff.values,
        x=pivot_diff.columns.values,
        y=pivot_diff.index.values,
        colorscale='RdBu',
        colorbar=dict(title='Difference'),
        hovertemplate='Moneyness=%{x:.2f}<br>T(days)=%{y:.2f}<br>Diff=%{z:.4f}<extra></extra>',
    )
)

heatmap.update_layout(
    title='Difference Heatmap (Actual - Predicted)',
    xaxis_title='Strike (K)',
    yaxis_title='Maturity (days)',
    height=520,
    width=1050,
)
heatmap.show()